In [1]:
from entsoe import EntsoePandasClient
import pandas as pd
from tqdm import tqdm
import time
import warnings
from typing import Sequence
import xarray as xr
import cfgrib

## ENTSO-e/ EMBER data:

- Need to register on https://transparency.entsoe.eu/ website and write an email to transparency@entsoe.eu in order to obtain a personal API key, which can be used with entsoe pandas client to retrieve data.
- **pip install entsoe-py** if not done yet
- Following data is needed:
    - **Generation data** -- electricity generation broken down by production type. Frequency -- 15 min intervals.
    - **Load data** -- electricity demand, same frequency intervals.
    - **Price data** -- electricity hourly prices. However, I found it tricky to get this data, due to different bidding zone naming and somehow missing availability before 2018. I use hourly prices from EMBER instead, https://ember-energy.org/data/european-wholesale-electricity-price-data/

Below is the general function to retrieve generation and load data. 

In [2]:
client = EntsoePandasClient(api_key="KEY") ## change with your key

def query_with_progress(client, country, method: str, start, end, freq="M"):
    """
    freq: 'M' (monthly) or 'W' (weekly) to avoid timeouts. This is NOT the data frequency, it is only related
    to retrieval.
    """
    if not hasattr(client, method):
        raise ValueError(f"Client has no method '{method}'")

    fn = getattr(client, method)
    
    chunks = pd.date_range(start, end, freq=freq, tz=start.tz)
    if chunks[-1] < end:
        chunks = chunks.append(pd.DatetimeIndex([end]))

    dfs = []

    for i in tqdm(range(len(chunks)-1), desc=f"{method} {country}"):
        s, e = chunks[i], chunks[i+1]
        try:
            df = fn(country, start=s, end=e)
            dfs.append(df)
        except Exception as ex:
            print(f"Failed {country} {s.date()} → {e.date()}: {ex}")

    return pd.concat(dfs).sort_index()

In [ ]:
# Define the data range and call the function

start = pd.Timestamp('2017-01-01', tz='Europe/Brussels')
end   = pd.Timestamp('2023-12-31', tz='Europe/Brussels')

## Generation for DE and FR
generation_de = query_with_progress(client, "DE", "query_generation", start, end, freq="M")
generation_fr = query_with_progress(client, "FR", "query_generation", start, end, freq="M")

# Load for DE and FR
load_de = query_with_progress(client, "DE","query_load", start, end, freq="M")
load_fr = query_with_progress(client, "FR","query_load", start, end, freq="M")

# Price data via general client, not the function above. Not recommended, better use EMBER data
prices_de = client.query_day_ahead_prices('DE_LU', start=start, end=end)
prices_fr = client.query_day_ahead_prices('FR', start=start, end=end)

### Data saver to parquet

Recommended to save the data immediately, to avoid accidental loss, because it takes time to retrieve. Parquet is a good format for the large datasets due to its high compression and fast query performance.

In [ ]:
def save_entsoe_v2(
    df: pd.DataFrame,
    path: str,
    tz_local: str = "Europe/Brussels",
    engines: Sequence[str] = ("fastparquet", "pyarrow"),
    convert_index_to_utc: bool = True,
    ambiguous = "NaT",
    nonexistent = "shift_forward",
    verbose: bool = True
) -> str:
    """
    Save ENTSO-e Dataframe to parquet.
    Tries engines in order; if do not work writes CSV.
    """
    df = df.copy()

    if isinstance(df, pd.Series):
        df = df.to_frame(name=df.name or "value")

    # -------------------------
    # index: localize / convert / strip tz
    # -------------------------
    if isinstance(df.index, pd.DatetimeIndex):
        try:
            if df.index.tz is None:
                df.index = df.index.tz_localize(
                    tz_local, ambiguous=ambiguous, nonexistent=nonexistent
                )
            if convert_index_to_utc:
                df.index = df.index.tz_convert("UTC").tz_localize(None)
        except Exception as e:
            # fallback attempt with safe args
            if verbose:
                print("Index localization error:", e)
                print("Retrying with ambiguous='NaT', nonexistent='shift_forward'")
            df.index = df.index.tz_localize(
                tz_local, ambiguous="NaT", nonexistent="shift_forward"
            )
            if convert_index_to_utc:
                df.index = df.index.tz_convert("UTC").tz_localize(None)

    # -------------------------
    # flatten / sanitize column names
    # -------------------------
    def _stringify_name(elem):
        if elem is None:
            return ""
        if isinstance(elem, (list, tuple)):
            return "_".join(str(x) for x in elem)
        return str(elem)

    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []
        for i, col in enumerate(df.columns):
            parts = []
            for c in col:
                parts.append(_stringify_name(c))
            joined = "_".join([p for p in parts if p != ""]).strip("_")
            if joined == "":
                joined = f"col_{i}"
            new_cols.append(joined)
        # ensure unique
        seen = {}
        final_cols = []
        for c in new_cols:
            if c in seen:
                seen[c] += 1
                final_cols.append(f"{c}_{seen[c]}")
            else:
                seen[c] = 0
                final_cols.append(c)
        df.columns = final_cols
    else:
        df.columns = [("" if c is None else str(c)) or f"col_{i}" for i, c in enumerate(df.columns)]

    # -------------------------
    # convert problematic dtypes (use isinstance checks per deprecation)
    # -------------------------
    for col in list(df.columns):
        dtype = df[col].dtype
        try:
            # PeriodDtype
            if isinstance(dtype, pd.PeriodDtype):
                if verbose:
                    print(f"Converting Period column -> timestamp: {col}")
                df[col] = df[col].dt.to_timestamp()
            # CategoricalDtype
            elif isinstance(dtype, pd.CategoricalDtype):
                if verbose:
                    print(f"Converting Categorical -> string: {col}")
                df[col] = df[col].astype(str)
            # Interval
            elif isinstance(dtype, pd.IntervalDtype):
                if verbose:
                    print(f"Converting Interval -> string: {col}")
                df[col] = df[col].astype(str)
            # Extension arrays (arrow extension etc.)
            elif pd.api.types.is_extension_array_dtype(dtype):
                if verbose:
                    print(f"Converting extension array dtype -> string: {col} (dtype={dtype})")
                df[col] = df[col].astype(str)
            # Object columns that hide lists/tuples/periods/timestamps
            elif pd.api.types.is_object_dtype(dtype):
                sample = df[col].dropna().head(10).tolist()
                if any(isinstance(x, (list, tuple, pd.Period, pd.Timestamp)) for x in sample):
                    if verbose:
                        print(f"Normalizing object column -> string: {col}")
                    df[col] = df[col].astype(str)
        except Exception as e:
            if verbose:
                print(f"Warning converting {col}: {e}. Coercing to string.")
            df[col] = df[col].astype(str)

    # -------------------------
    # final safety pass: coerce remaining extension arrays to string
    # -------------------------
    for col in df.columns:
        if pd.api.types.is_extension_array_dtype(df[col].dtype):
            if verbose:
                print(f"Final coercion to string for extension column: {col}")
            df[col] = df[col].astype(str)

    # -------------------------
    # try parquet engines in order; handle ArrowKeyError
    # -------------------------
    last_exc = None
    for engine in engines:
        try:
            if verbose:
                print(f"Trying to write with engine='{engine}' -> {path}")
            # small safety: suppress pyarrow extension noisy logs if present
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                df.to_parquet(path, engine=engine, index=True)
            if verbose:
                print("Write successful with engine:", engine)
            return path
        except Exception as e:
            last_exc = e
            msg = repr(e).lower()
            if "arrowkeyerror" in msg or "pandas.period" in msg or "type extension" in msg:
                # PyArrow extension registration problem
                print("\nPyArrow extension registration error detected.")
                print("This often happens when PyArrow has stale extension registrations in this Python session.")
                if engine == "pyarrow":
                    print("Recommended action: restart your kernel / python process and re-run. Falling back to next engine.")
                else:
                    print(f"Engine '{engine}' failed with Arrow-related error; trying next engine.")
                continue
            else:
                if verbose:
                    print(f"Engine '{engine}' failed with: {e}. Trying next fallback (if any).")
                continue

    # -------------------------
    # Final fallback: write CSV
    # -------------------------
    csv_path = path.rsplit(".", 1)[0] + ".csv"
    try:
        if verbose:
            print(f"All parquet engines failed. Writing CSV to {csv_path} (uncompressed).")
        df.to_csv(csv_path, index=True)
        return csv_path
    except Exception as e:
        # give a helpful error
        raise RuntimeError("All write attempts failed; last error: ") from last_exc

In [ ]:
# Use the saver function 
save_entsoe_v2(generation_de, "generation_DE_17-23.parquet", verbose=True)
save_entsoe_v2(generation_fr, "generation_FR_17-23.parquet", verbose=True)

save_entsoe_v2(load_de, "load_DE_17-23.parquet", verbose=True)
save_entsoe_v2(load_fr, "load_FR_17-23.parquet", verbose=True)

# To load the data back, same for the other files
dfgen_DE = pd.read_parquet('generation_DE_17-23.parquet',engine="fastparquet")

## Copernicus weather data

- Again, registration is required. The data is here: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download
- After registration is completed, data queries can be made. Data is large, so needs to be split to 2-3 years per query. Data is in grib format, so **pip install cfgrib** if not already done.
- Request data like this:
    - Product type: reanalysis
    - Data format: GRIB
    - Variable: 10m u-component of wind, 10m v-component of wind, 2m temperature, Total precipitation
    - Month: select all
    - Day: select all
    - Time: select all
    - Year: 2-3 years at a time
    - Sub-region extraction, for Germany + France: North: 55°, West: -5°, South: 42°, East: 15°
- Then calculate wind speed as $\sqrt{u^2 + v^2}$ (will be shown in the code below)

## GRIB data extraction below

In [ ]:
grib_files = [
    "weather_17-18.grib",
    "weather_19-20.grib",
    "weather_21-23.grib"
]

# Bounding country area boxes (roughly)
COUNTRY_BBOX = {
    "DE": dict(lat_min=47, lat_max=55, lon_min=5, lon_max=15),
    "FR": dict(lat_min=42, lat_max=51, lon_min=-5, lon_max=8),
}

# -------------------------------------------------
# Process GRIB file
# -------------------------------------------------

def process_grib_file(filepath):
    print(f"\nProcessing {filepath}")
    
    datasets = cfgrib.open_datasets(filepath)
    
    # Separate main surface dataset (u10, v10, t2m)
    surface = None
    precip = None
    
    for ds in datasets:
        if "u10" in ds.data_vars:
            surface = ds
        if "tp" in ds.data_vars:
            precip = ds
    
    # --- compute wind speed ---
    wind = np.sqrt(surface["u10"]**2 + surface["v10"]**2)
    surface = surface.assign(wind=wind)
    
    # --- fix precipitation dataset ---
    if precip is not None:
        precip = precip.mean(dim="step")  # collapse forecast step
        surface = surface.merge(precip)
    
    return surface


# -------------------------------------------------
# country spatial average
# -------------------------------------------------

def extract_country(ds, country):
    bbox = COUNTRY_BBOX[country]
    
    subset = ds.sel(
        latitude=slice(bbox["lat_max"], bbox["lat_min"]),
        longitude=slice(bbox["lon_min"], bbox["lon_max"])
    )
    
    # spatial mean
    subset_mean = subset.mean(dim=["latitude", "longitude"])
    
    return subset_mean


# -------------------------------------------------
# MAIN 
# -------------------------------------------------

all_de = []
all_fr = []

for file in grib_files:
    ds = process_grib_file(file)
    
    de = extract_country(ds, "DE")
    fr = extract_country(ds, "FR")
    
    # convert to pandas
    df_de = de.to_dataframe().reset_index()
    df_fr = fr.to_dataframe().reset_index()
    
    all_de.append(df_de)
    all_fr.append(df_fr)

# concatenate all years
df_de = pd.concat(all_de).sort_values("time")
df_fr = pd.concat(all_fr).sort_values("time")

# -------------------------------------------------
# Daily resampling, final data is df_de and df_fr
# -------------------------------------------------

df_de["time"] = pd.to_datetime(df_de["time"])
df_fr["time"] = pd.to_datetime(df_fr["time"])

df_de = df_de.set_index("time").resample("D").mean()
df_fr = df_fr.set_index("time").resample("D").mean()

## Final aggregated data files

After all the data is available, I generated aggregated data files containing all the per-country information, with daily frequency, see below

In [ ]:
# ================================
# SETTINGS (CHANGE PER COUNTRY)
# ================================
country_name = "France"   # Change to "Germany" when running for Germany

# EMBER hourly price file
price_file = "european_wholesale_electricity_price_data_hourly/France.csv"

output_file = f"daily_merged_{country_name.lower()}.csv"

# =====================================================
# Force clean daily datetime index
# =====================================================
def force_daily_index(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index)

    # Remove timezone if present
    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)

    # Remove time component
    df.index = df.index.normalize()

    return df

# =====================================================
# PRICES → LOG PRICES AND DAILY REALIZED VOLATILITY
# =====================================================
prices = pd.read_csv(price_file, parse_dates=['Datetime (Local)'])

prices = prices[['Datetime (Local)', 'Price (EUR/MWhe)']].copy()
prices.columns = ['datetime', 'price']
prices = prices.sort_values('datetime')
prices.set_index('datetime', inplace=True)

prices['log_price'] = np.log(prices['price'].replace(0, np.nan))
prices['hourly_ret'] = prices['log_price'].diff()

prices['date'] = prices.index.normalize()

daily_vol = prices.groupby('date')['hourly_ret'].apply(
    lambda x: np.sqrt(np.sum(x.dropna()**2))
).to_frame(name='realized_vol')

daily_close = prices['log_price'].resample('D').last()
daily_vol['daily_log_return'] = daily_close.diff()

daily_vol = force_daily_index(daily_vol)

# =====================================================
# WEATHER (already loaded from parquet as df_de)
# =====================================================
weather = df_fr.copy()

weather = weather[['t2m', 'wind', 'tp']]
weather.index = pd.to_datetime(weather.index)
weather.index = weather.index.normalize()

# Kelvin → Celsius
weather['t2m'] = weather['t2m'] - 273.15
weather.columns = ['temp_C', 'wind_speed', 'precip']

# Aggregate if duplicates exist
weather = weather.groupby(weather.index).mean()
weather = force_daily_index(weather)

# =====================================================
# GENERATION (already loaded, 15-min → daily mean)
# =====================================================
generation = dfgen_FR.copy()
generation.index = pd.to_datetime(generation.index)

generation_daily = generation.resample('D').mean()
generation_daily = force_daily_index(generation_daily)

# Optional: Create useful aggregates (recommended)
generation_daily['total_generation'] = generation_daily.sum(axis=1)

if 'Wind Onshore_Actual Aggregated' in generation_daily.columns:
    generation_daily['wind_total'] = (
        generation_daily.get('Wind Onshore_Actual Aggregated', 0)
        + generation_daily.get('Wind Offshore_Actual Aggregated', 0)
    )
    generation_daily['wind_share'] = (
        generation_daily['wind_total'] /
        generation_daily['total_generation']
    )

if 'Solar_Actual Aggregated' in generation_daily.columns:
    generation_daily['solar_total'] = generation_daily['Solar_Actual Aggregated']
    generation_daily['solar_share'] = (
        generation_daily['solar_total'] /
        generation_daily['total_generation']
    )

# =====================================================
# LOAD (15-min → daily mean)
# =====================================================
load = dfload_FR.copy()
load.index = pd.to_datetime(load.index)

load_daily = load.resample('D').mean()
load_daily.columns = ['load_MW']
load_daily = force_daily_index(load_daily)

# =====================================================
# EXPLICIT COMMON DATE WINDOW
# =====================================================
start_date = max(
    daily_vol.index.min(),
    weather.index.min(),
    generation_daily.index.min(),
    load_daily.index.min()
)

end_date = min(
    daily_vol.index.max(),
    weather.index.max(),
    generation_daily.index.max(),
    load_daily.index.max()
)

print("Common date range:", start_date, "to", end_date)

daily_vol = daily_vol.loc[start_date:end_date]
weather = weather.loc[start_date:end_date]
generation_daily = generation_daily.loc[start_date:end_date]
load_daily = load_daily.loc[start_date:end_date]

# =====================================================
# MERGE (and keep NaNs in exogenous variables)
# =====================================================
df = daily_vol.join(weather, how='left')
df = df.join(generation_daily, how='left')
df = df.join(load_daily, how='left')

# Only require target variables to exist
df = df.dropna(subset=['realized_vol', 'daily_log_return'])

print(f"{country_name} final dataset shape:", df.shape)
print("Final date range:", df.index.min(), "to", df.index.max())

# =====================================================
# SAVE
# =====================================================
df.to_csv(output_file)

print(f"Saved merged dataset to: {output_file}")